In [1]:
from playerPredictor import get_team_player_scores, identify_nba_season
from teamPredictor import calculate_net_four, calculate_recent_form_scores
# identify_nba_season('12/22/2020')

In [3]:
def compute_team_strength_score(engine, date, team_name):
    # compute team performance scores
    recent_form = calculate_recent_form_scores(engine, date, team_name)
    season_net, season_four_factors = calculate_net_four(engine, date, team_name)

    team_performance = (0.7 * recent_form) + (0.3 * (0.5 * season_net + 0.5 * season_four_factors))

    # computer player contribution score
    player_scores_df = get_team_player_scores(engine, team_name, date)

    star_players = player_scores_df[player_scores_df["CATEGORY"] == "Star"]
    rotation_players = player_scores_df[player_scores_df["CATEGORY"] == "Rotation"]

    star_score = star_players["SCORE"].mean()
    rotation_score = rotation_players["SCORE"].mean()

    player_contribution = (0.7 * star_score) + (0.3 * rotation_score)

    # Computer Historical Matchup Scores
    # historical_matchups = calculate_historic_matchups(engine, date, team_name, opponent_team, avg_games=3, weighted=True)

    # historical_matchup_score = (0.5 * historical_matchups['net_rating_diff']) + \
    #                            (0.3 * historical_matchups['efg_diff']) + \
    #                            (0.2 * historical_matchups['turnover_pct_diff'])

    TSS = (0.5 * team_performance) + (0.5 * player_contribution)  # + (0.15 * historical_matchup_score)
    return TSS

In [27]:
import os
from datetime import datetime, timedelta

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)
team_name = "Detroit Pistons"


# date = '03-13-2023'
# date = '2024-12-13'
# compute_team_strength_score(engine, date, team_name)
def track_team_strength_over_time(
    engine,
    team_name,
    start_date_str="2023-11-01",
    end_date_str="2024-03-01",
    step_days=5,
    season_start_dates=None,
):
    if season_start_dates is None:
        season_start_dates = {
            "2020-21": "2020-12-22",
            "2021-22": "2021-10-19",
            "2022-23": "2022-10-18",
            "2023-24": "2023-10-24",
            "2024-25": "2024-10-22",
        }

    # Convert date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")

    # Initialize results list to store data
    results = []

    # Initialize current date
    current_date = start_date

    print(f"Analyzing {team_name} strength from {start_date_str} to {end_date_str} every {step_days} days...")

    # Loop through dates
    while current_date <= end_date:
        try:
            # Format date for the compute_team_strength_score function
            date_str = current_date.strftime("%Y-%m-%d")

            print(f"Computing strength score for {team_name} on {date_str}...")

            # Compute team strength score
            strength_score = compute_team_strength_score(engine, date_str, team_name)

            # Get NBA season for the date
            season = identify_nba_season(date_str)

            # Store result
            results.append(
                {
                    "Team": team_name,
                    "Date": date_str,
                    "Strength_Score": strength_score,
                    "Season": season,
                }
            )

            print(f"  Score: {strength_score:.4f}")

        except Exception as e:
            print(f"Error computing strength score for {date_str}: {e}")
            # Add a row with error information
            results.append(
                {
                    "Team": team_name,
                    "Date": date_str,
                    "Strength_Score": None,
                    "Season": identify_nba_season(date_str),
                    "Error": str(e),
                }
            )

        # Move to next date
        current_date += timedelta(days=step_days)

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)

    # Display summary
    if not results_df.empty:
        print("\nSummary Statistics:")
        print(f"Mean Strength Score: {results_df['Strength_Score'].mean():.4f}")
        print(f"Min Strength Score: {results_df['Strength_Score'].min():.4f}")
        print(f"Max Strength Score: {results_df['Strength_Score'].max():.4f}")

    return results_df

In [41]:
start_date_str = "2023-11-01"
end_date_str = "2024-03-01"
# Convert date strings to datetime objects
start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
print(start_date)
print(end_date)
date_str = start_date.strftime("%Y-%m-%d")
print(date_str)

2023-11-01 00:00:00
2024-03-01 00:00:00
2023-11-01


In [45]:
import os

from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)
team_name = "Detroit Pistons"
results = track_team_strength_over_time(
    engine, team_name, start_date_str="2024-11-01", end_date_str="2025-03-01", step_days=5
)
results

Analyzing Detroit Pistons strength from 2024-11-01 to 2025-03-01 every 5 days...
Computing strength score for Detroit Pistons on 2024-11-01...
Error computing strength score for 2024-11-01: list index out of range
Computing strength score for Detroit Pistons on 2024-11-06...


C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:178: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)


Error computing strength score for 2024-11-06: list index out of range
Computing strength score for Detroit Pistons on 2024-11-11...


C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:178: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)


Error computing strength score for 2024-11-11: list index out of range
Computing strength score for Detroit Pistons on 2024-11-16...


...

C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:178: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)


Net Rating Score: 13.748000000000001
Recent Four Factors Score: nan


C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:364: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)
C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:364: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)
C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:364: FutureWarnin

Net Rating Score: 1.098181818181818
Recent Four Factors Score: nan
  Score: nan

Summary Statistics:
Mean Strength Score: 1.0678
Min Strength Score: -1.3998
Max Strength Score: 3.9800


,Team,Date,Strength_Score,Season,Error
0,Detroit Pistons,2024-11-01,NaN,2024-25,list index out of range
1,Detroit Pistons,2024-11-06,NaN,2024-25,list index out of range
2,Detroit Pistons,2024-11-11,NaN,2024-25,list index out of range
3,Detroit Pistons,2024-11-16,NaN,2024-25,list index out of range
4,Detroit Pistons,2024-11-21,1.677744,2024-25,NaN
5,Detroit Pistons,2024-11-26,1.560229,2024-25,NaN
6,Detroit Pistons,2024-12-01,0.669956,2024-25,NaN
7,Detroit Pistons,2024-12-06,-0.645692,2024-25,NaN
8,Detroit Pistons,2024-12-11,-0.036002,2024-25,NaN
9,Detroit Pistons,2024-12-16,-1.399784,2024-25,NaN


In [47]:
import os

from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)
team_name = "Atlanta Hawks"
results_2 = track_team_strength_over_time(
    engine, team_name, start_date_str="2024-11-01", end_date_str="2025-03-01", step_days=5
)
results_2

Analyzing Atlanta Hawks strength from 2024-11-01 to 2025-03-01 every 5 days...
Computing strength score for Atlanta Hawks on 2024-11-01...
Error computing strength score for 2024-11-01: list index out of range
Computing strength score for Atlanta Hawks on 2024-11-06...
Error computing strength score for 2024-11-06: list index out of range
Computing strength score for Atlanta Hawks on 2024-11-11...
Error computing strength score for 2024-11-11: list index out of range
Computing strength score for Atlanta Hawks on 2024-11-16...
Error computing strength score for 2024-11-16: list index out of range
Computing strength score for Atlanta Hawks on 2024-11-21...
Net Rating Score: -3.062
Recent Four Factors Score: 0.48753500000000005
Net Rating Score: -4.65625
Recent Four Factors Score: -3.390959375
  Score: -0.0213
Computing strength score for Atlanta Hawks on 2024-11-26...
Net Rating Score: -6.394
Recent Four Factors Score: 0.48779600000000006
Net Rating Score: -5.466666666666667
Recent Four 

C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:178: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)


Net Rating Score: 0.526
Recent Four Factors Score: nan


C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:364: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)


Net Rating Score: -2.2877192982456136
Recent Four Factors Score: nan
  Score: nan
Computing strength score for Atlanta Hawks on 2025-03-01...


C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:178: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)


Net Rating Score: 0.526
Recent Four Factors Score: nan


C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:364: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)


Net Rating Score: -2.2877192982456136
Recent Four Factors Score: nan
  Score: nan

Summary Statistics:
Mean Strength Score: 0.5693
Min Strength Score: -1.0477
Max Strength Score: 2.4314


,Team,Date,Strength_Score,Season,Error
0,Atlanta Hawks,2024-11-01,NaN,2024-25,list index out of range
1,Atlanta Hawks,2024-11-06,NaN,2024-25,list index out of range
2,Atlanta Hawks,2024-11-11,NaN,2024-25,list index out of range
3,Atlanta Hawks,2024-11-16,NaN,2024-25,list index out of range
4,Atlanta Hawks,2024-11-21,-0.021336,2024-25,NaN
5,Atlanta Hawks,2024-11-26,-0.968036,2024-25,NaN
6,Atlanta Hawks,2024-12-01,1.141123,2024-25,NaN
7,Atlanta Hawks,2024-12-06,2.431382,2024-25,NaN
8,Atlanta Hawks,2024-12-11,1.440873,2024-25,NaN
9,Atlanta Hawks,2024-12-16,1.130751,2024-25,NaN


In [49]:
import os

from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)
team_name = "Cleveland Cavaliers"
results_2 = track_team_strength_over_time(
    engine, team_name, start_date_str="2024-11-01", end_date_str="2025-03-01", step_days=5
)
results_2

Analyzing Cleveland Cavaliers strength from 2024-11-01 to 2025-03-01 every 5 days...
Computing strength score for Cleveland Cavaliers on 2024-11-01...
Error computing strength score for 2024-11-01: list index out of range
Computing strength score for Cleveland Cavaliers on 2024-11-06...


C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:178: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)


Error computing strength score for 2024-11-06: list index out of range
Computing strength score for Cleveland Cavaliers on 2024-11-11...


C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:178: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)


Error computing strength score for 2024-11-11: list index out of range
Computing strength score for Cleveland Cavaliers on 2024-11-16...


...

C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:178: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)


Net Rating Score: 16.870000000000005
Recent Four Factors Score: nan


C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:364: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)
C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:364: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)


Net Rating Score: 10.723214285714286
Recent Four Factors Score: nan
  Score: nan

Summary Statistics:
Mean Strength Score: 4.8726
Min Strength Score: 2.6496
Max Strength Score: 6.7296


,Team,Date,Strength_Score,Season,Error
0,Cleveland Cavaliers,2024-11-01,NaN,2024-25,list index out of range
1,Cleveland Cavaliers,2024-11-06,NaN,2024-25,list index out of range
2,Cleveland Cavaliers,2024-11-11,NaN,2024-25,list index out of range
3,Cleveland Cavaliers,2024-11-16,NaN,2024-25,list index out of range
4,Cleveland Cavaliers,2024-11-21,5.083306,2024-25,NaN
5,Cleveland Cavaliers,2024-11-26,5.400863,2024-25,NaN
6,Cleveland Cavaliers,2024-12-01,3.451137,2024-25,NaN
7,Cleveland Cavaliers,2024-12-06,3.942833,2024-25,NaN
8,Cleveland Cavaliers,2024-12-11,4.418143,2024-25,NaN
9,Cleveland Cavaliers,2024-12-16,4.538598,2024-25,NaN
